# Agentic AI Pipeline — Task-Routing Agent

**Goal:** Build an `agent(query)` function that routes a natural-language query to the
correct tool (`calculator` or `extract_keywords`) based on simple keyword checks, and
always returns a clean JSON-style response:

```
{"type": "calculation" / "keywords" / "general" / "error", "result": ...}
```


## 1. Baseline tools

`calculator` and `extract_keywords` — the two pre-built tools the agent will route to.

In [1]:
import json
import re

def calculator(expression: str):
    """Safely evaluate a basic math expression (digits, + - * / ( ) only)."""
    if not re.fullmatch(r"[0-9\.\+\-\*\/\(\)\s]+", expression):
        raise ValueError("Unsafe or invalid expression")
    return eval(expression, {"__builtins__": {}}, {})


def extract_keywords(text: str):
    """Simple keyword extractor: lowercases, strips stopwords, returns unique words."""
    stopwords = {"the", "a", "an", "is", "are", "of", "in", "on",
                 "and", "to", "for", "this", "that", "about"}
    words = re.findall(r"[a-zA-Z']+", text.lower())
    return [w for w in dict.fromkeys(words) if w not in stopwords]


## 2. The agent — task-routing pipeline

Routing rules:
- contains `"calculate"` → parse the math expression → `calculator()` → `type: "calculation"`
- contains `"keywords"` → pipe text → `extract_keywords()` → `type: "keywords"`
- anything else → fallback text response → `type: "general"`
- anything that fails to parse / errors out → `type: "error"`, never an unhandled crash

In [2]:
def agent(query: str) -> dict:
    """
    Routes a natural-language query to the correct tool based on
    keyword pattern checks, and returns a structured JSON-style dict:
        {"type": "calculation" | "keywords" | "general" | "error", "result": ...}
    """
    if not isinstance(query, str) or not query.strip():
        return {"type": "error", "result": "Empty or invalid input."}

    q_lower = query.lower()

    try:
        # --- Route 1: calculation ---
        if "calculate" in q_lower:
            after = query.lower().split("calculate", 1)[1]
            match = re.search(r"[\d\.\(\)\s\+\-\*\/]+", after)
            if not match or not match.group(0).strip():
                return {"type": "error", "result": "No valid math expression found after \'calculate\'."}
            expression = match.group(0).strip()
            result = calculator(expression)
            return {"type": "calculation", "result": result}

        # --- Route 2: keyword extraction ---
        elif "keywords" in q_lower:
            text = query.lower().replace("keywords", "").strip()
            if not text:
                return {"type": "error", "result": "No text supplied for keyword extraction."}
            keywords = extract_keywords(text)
            return {"type": "keywords", "result": keywords}

        # --- Route 3: fallback / general ---
        else:
            return {"type": "general", "result": f"I received your message: \'{query}\'. No specific tool matched."}

    except ZeroDivisionError:
        return {"type": "error", "result": "Division by zero."}
    except Exception as e:
        return {"type": "error", "result": f"Failed to process query: {str(e)}"}


## 3. Automated validation checks

Runs a fixed array of test queries (including edge cases / bad input) and confirms every response is valid, structured JSON with both required keys.

In [3]:
test_queries = [
    "calculate 12 + 8 * 2",
    "give me keywords from this sentence about machine learning agents",
    "hello, how are you?",
    "calculate",              # missing expression -> error
    "calculate 5 / 0",        # divide by zero -> error
    "",                       # empty input -> error
    "keywords",                # no text supplied -> error
]

print("=== Running automated validation checks ===\n")
for tq in test_queries:
    output = agent(tq)
    print(f"Query: {tq!r}")
    print(json.dumps(output, indent=2))
    print("-" * 40)
    assert isinstance(output, dict) and "type" in output and "result" in output, "Missing required keys!"

print("\nAll validation checks passed structurally.")


=== Running automated validation checks ===

Query: 'calculate 12 + 8 * 2'
{
  "type": "calculation",
  "result": 28
}
----------------------------------------
Query: 'give me keywords from this sentence about machine learning agents'
{
  "type": "keywords",
  "result": [
    "give",
    "me",
    "from",
    "sentence",
    "machine",
    "learning",
    "agents"
  ]
}
----------------------------------------
Query: 'hello, how are you?'
{
  "type": "general",
  "result": "I received your message: 'hello, how are you?'. No specific tool matched."
}
----------------------------------------
Query: 'calculate'
{
  "type": "error",
  "result": "No valid math expression found after 'calculate'."
}
----------------------------------------
Query: 'calculate 5 / 0'
{
  "type": "error",
  "result": "Division by zero."
}
----------------------------------------
Query: ''
{
  "type": "error",
  "result": "Empty or invalid input."
}
----------------------------------------
Query: 'keywords'
{
  "

## 4. Interactive loop

Type queries live and see the structured JSON response. Type `exit` to stop.

In [ ]:
while True:
    user_input = input("Enter a query (or 'exit' to quit): ")
    if user_input.strip().lower() == "exit":
        print("Session ended.")
        break
    response = agent(user_input)
    print(json.dumps(response, indent=2))


{
  "type": "calculation",
  "result": 42
}
{
  "type": "keywords",
  "result": [
    "give",
    "me",
    "from",
    "document",
    "climate",
    "change",
    "renewable",
    "energy"
  ]
}
{
  "type": "general",
  "result": "I received your message: 'what is your name'. No specific tool matched."
}
